# Agent Monitor (Phase 4)

Subscribes to telemetry and roadwork events, computes congestion streaks, and publishes authoritative congestion messages.

In [ ]:
# Cell purpose: Connect to MQTT, subscribe to telemetry/roadwork, and initialize monitor state.
from collections import defaultdict
import json

from simulated_city import mqtt
from simulated_city.config import load_config
from simulated_city.metrics import count_cars_per_segment, update_congestion_streaks
from simulated_city.topics import cars_telemetry_topic, roadwork_events_topic, traffic_congestion_topic

config = load_config()
phase1 = config.simulation.car_rerouting_phase1 if config.simulation else None
if phase1 is None:
    raise ValueError("Missing simulation.car_rerouting_phase1 in config.yaml")

client = mqtt.connect_mqtt(config.mqtt, client_id_suffix="agent-monitor-phase4")
telemetry_topic = cars_telemetry_topic(config.mqtt.base_topic)
roadwork_topic = roadwork_events_topic(config.mqtt.base_topic)
congestion_topic = traffic_congestion_topic(config.mqtt.base_topic)

latest_tick = 0
latest_roadwork = {"active": False, "blocked_segment_ids": list(phase1.roadwork.blocked_segment_ids)}
telemetry_by_tick = defaultdict(list)
streaks = {}

def _subscribe_monitor_topics():
    client.subscribe(telemetry_topic)
    client.subscribe(roadwork_topic)

def _on_message(client_obj, userdata, msg):
    global latest_tick, latest_roadwork
    payload = json.loads(msg.payload.decode("utf-8"))
    if msg.topic == telemetry_topic:
        telemetry_by_tick[int(payload["tick"])].append(payload)
        latest_tick = max(latest_tick, int(payload["tick"]))
    elif msg.topic == roadwork_topic:
        latest_roadwork = payload

_previous_on_connect = getattr(client, "on_connect", None)
def _on_connect(client_obj, userdata, flags, reason_code, properties=None):
    if callable(_previous_on_connect):
        _previous_on_connect(client_obj, userdata, flags, reason_code, properties)
    _subscribe_monitor_topics()

client.on_connect = _on_connect
client.on_message = _on_message
_subscribe_monitor_topics()

print(f"Connected MQTT target: {config.mqtt.host}:{config.mqtt.port}")
print(f"Subscribed topics: {telemetry_topic}, {roadwork_topic}")
print(f"Publish topic: {congestion_topic}")

Disconnected from MQTT broker (reason=Unspecified error). Reconnecting...


Connected MQTT target: 127.0.0.1:1883
Subscribed topics: simulated-city/city/cars/telemetry, simulated-city/city/roadwork/events
Publish topic: simulated-city/city/traffic/congestion


Disconnected from MQTT broker (reason=Unspecified error). Reconnecting...


In [ ]:
# Cell purpose: Process telemetry into cars_per_segment and publish congestion state.
if not client.is_connected():
    client.reconnect()

processed_tick = latest_tick or 1
batch = telemetry_by_tick.get(processed_tick, [])
cars_per_segment = count_cars_per_segment(batch)
streaks, congested_segments = update_congestion_streaks(cars_per_segment, streaks, threshold=10, required_ticks=3)

payload = {
    "agent": "agent_monitor",
    "tick": processed_tick,
    "cars_per_segment": cars_per_segment,
    "congested_segment_ids": congested_segments,
    "roadwork_active": bool(latest_roadwork.get("active", False)),
    "blocked_segment_ids": latest_roadwork.get("blocked_segment_ids", []),
}
mqtt.publish_json_checked(client, congestion_topic, payload)

print(
    f"Published congestion tick={processed_tick}: segments={cars_per_segment}, "
    f"congested={congested_segments}"
)

Disconnected from MQTT broker (reason=Unspecified error). Reconnecting...
Disconnected from MQTT broker (reason=Unspecified error). Reconnecting...
Disconnected from MQTT broker (reason=Unspecified error). Reconnecting...
Disconnected from MQTT broker (reason=Unspecified error). Reconnecting...


RuntimeError: Publish verification timed out for topic 'simulated-city/city/traffic/congestion'

Disconnected from MQTT broker (reason=Unspecified error). Reconnecting...
Disconnected from MQTT broker (reason=Unspecified error). Reconnecting...
Disconnected from MQTT broker (reason=Unspecified error). Reconnecting...
Disconnected from MQTT broker (reason=Unspecified error). Reconnecting...
Disconnected from MQTT broker (reason=Unspecified error). Reconnecting...
Disconnected from MQTT broker (reason=Unspecified error). Reconnecting...
Disconnected from MQTT broker (reason=Unspecified error). Reconnecting...
Disconnected from MQTT broker (reason=Unspecified error). Reconnecting...
Disconnected from MQTT broker (reason=Unspecified error). Reconnecting...
Disconnected from MQTT broker (reason=Unspecified error). Reconnecting...
Disconnected from MQTT broker (reason=Unspecified error). Reconnecting...
Disconnected from MQTT broker (reason=Unspecified error). Reconnecting...
Disconnected from MQTT broker (reason=Unspecified error). Reconnecting...
Disconnected from MQTT broker (reason=

In [ ]:
# Cell purpose: Disconnect monitor MQTT client cleanly when done.
client.loop_stop()
client.disconnect()
print("Disconnected MQTT client.")